# PMAG (Protein Metal Affinity GCN)

This project is aimed at calculating the metal affinity for proteins in various bacteria. It uses the BindingDB database and a Graph Convolutional Network. Each molecule is expressed as a graph where the edges are molecules bonded to each other and each data point is a molecule.

## Processing Data

The .sdf file first has to be turned into a graph with the features, where the graph is a torch_geometric.data.Data object, containing the node and edge tables. The node features are

- element = a one-hot encoding of the element
- hybridization = the angle & shape of the bond the atom prefers
- charge = the formal charge of the atom

The edge features are

- bond type = how strong the bond is
- aromaticity = whether the electrons can change between the atoms

In [ ]:
from networkx.readwrite.json_graph import adjacency
# Loading molecules
from rdkit import Chem
from torchgen.gen import file_manager_from_dispatch_key

path_mols = 'data/BindingDB_All_2D.sdf'
i = 0

with Chem.MultithreadedSDMolSupplier(path_mols) as suppl:
  for mol in suppl:
    if mol is None: continue

    if i % 100 == 0:
        print(mol.GetNumAtoms())

In [1]:
# Calculating a feature - replace GetHybridization() with whatever
from rdkit import Chem

def grab_first_valid(mols):
    for mol in mols:
        if mol != None: return mol

path_mols = 'data/BindingDB_All_2D.sdf'
mols = Chem.SDMolSupplier(path_mols)
print('Loaded')

mol = grab_first_valid(mols)
Chem.SanitizeMol(mol)
print('Put on sanitizer')

for atom in mol.GetAtoms():
    print(atom.GetHybridization())

Loaded
Put on sanitizer
SP2
SP2
SP2
SP3
SP3
SP3
SP3
SP2
SP2
SP2
SP2
SP2
SP2
SP2
SP3
SP3
SP2
SP2
SP2
SP2
SP2
SP2
SP2
SP2
SP3
SP3
SP3
SP3
SP2
SP2
SP2
SP2
SP2
SP2
SP2
SP2
SP2
SP2
SP2
SP2


[12:25:27] ERROR: Cannot convert '-1.' to unsigned int on line 4
[12:25:27] ERROR: moving to the beginning of the next molecule


In [ ]:
# Run workers.py due to IPython multiprocessing issues
import numpy as np
import torch
from torch_geometric.data import Data
import tqdm
import multiprocessing as mp
from rdkit import Chem

def get_chunks(file_path, chunk_size=1000):
    supplier = Chem.SDMolSupplier(file_path)
    chunk = []

    for mol in supplier:
        chunk.append(mol)

        if len(chunk) == chunk_size:
            yield chunk
            chunk = []

def process(mol):
    X = []
    edge_features = []

    encodings = {}
    encoded = None
    max_encode = 0

    edge_matrix_idx1 = []
    edge_matrix_idx2 = []
    adjacency_matrix_idx1 = []
    adjacency_matrix_idx2 = []
    adjacency_matrix = []

    with tqdm.tqdm(unit='mol'):
        if mol is None: return None

        for atom in mol.GetAtoms():
            for encoding, value in encodings.items():
                if value == atom.GetSymbol():
                    encoded = encoding
                else:
                    encoded = max_encode + 1
                    encoding.append(encoded, atom.GetSymbol())
                    max_encode += 1

            X.append([encoded, atom.GetHybridization(), atom.GetFormalCharge()])

        for bond in mol.GetBonds():
            edge_features.append([bond.GetBondType().real, float(bond.GetIsAromatic())])

            edge_matrix_idx1.append(bond.GetBeginAtomIdx())
            edge_matrix_idx2.append(bond.GetEndAtomIdx())

            edge_matrix_idx2.append(bond.GetBeginAtomIdx())
            edge_matrix_idx1.append(bond.GetEndAtomIdx())

    edge_matrix = np.array(adjacency_matrix.append(adjacency_matrix_idx1, adjacency_matrix_idx2))
    X = np.array(X)
    edge_features = np.array(edge_features)

    return(Data(x=X, edge_index=edge_matrix, edge_attr=edge_features))

index = 0

if __name__ == '__main__':
    mols_path = 'data/BindingDB_All_2D.sdf'

    with mp.Pool(processes=8) as pool:
        results = pool.imap(process, get_chunks(mols_path), chunksize=200)

        for res in results:
            torch.save(res, f'./data/graphs_chunks/mol_chunk_{index:04d}.pt')